# LLM PKM Harness — Proof of Concept

Full implementation of the compiled-wiki harness with **real Ollama LLM and embedding API calls**.

**Prerequisites:** Ollama running locally with models pulled:
```bash
ollama serve   # if not already running
ollama pull nomic-embed-text
ollama pull qwen2.5:7b
```

**Run:** `uv sync && uv run jupyter notebook harness_poc.ipynb`

Pipeline: `raw/` → **LLM ingest** → `wiki/` + embeddings → **semantic query** → **LLM answer** → lint

## 1. Configuration & Ollama client

In [ ]:
from __future__ import annotations

import json
import re
import shutil
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Any

import httpx
import numpy as np

# --- Config (override for remote Ollama) ---
OLLAMA_HOST = "http://localhost:11434"
CHAT_MODEL = "qwen2.5:7b"          # ingest extraction + query synthesis
EMBED_MODEL = "nomic-embed-text"   # semantic retrieval

POC_DIR = Path.cwd() if Path.cwd().name == "poc" else Path("researches/llm_harnessing_for_pkm/poc").resolve()
VAULT_ROOT = POC_DIR / "sample_vault"

WIKILINK_RE = re.compile(r"\[\[([^\]|]+)(?:\|[^\]]+)?\]\]")
INDEX_ENTRY_RE = re.compile(r"^- \[\[(.+?)\]\] — (.+)$")


def ollama_chat(messages: list[dict[str, str]], *, model: str = CHAT_MODEL) -> str:
    """Call Ollama /api/chat (non-streaming)."""
    r = httpx.post(
        f"{OLLAMA_HOST}/api/chat",
        json={"model": model, "messages": messages, "stream": False},
        timeout=180.0,
    )
    r.raise_for_status()
    return r.json()["message"]["content"]


def ollama_embed(texts: list[str], *, model: str = EMBED_MODEL) -> list[list[float]]:
    """Call Ollama /api/embed."""
    r = httpx.post(
        f"{OLLAMA_HOST}/api/embed",
        json={"model": model, "input": texts},
        timeout=120.0,
    )
    r.raise_for_status()
    return r.json()["embeddings"]


def check_ollama() -> None:
    try:
        tags = httpx.get(f"{OLLAMA_HOST}/api/tags", timeout=5.0).json().get("models", [])
        names = {m["name"].split(":")[0] for m in tags}
        for need in (CHAT_MODEL.split(":")[0], EMBED_MODEL.split(":")[0]):
            if need not in names:
                raise RuntimeError(f"Model '{need}' not found. Run: ollama pull {need}")
        print(f"Ollama OK at {OLLAMA_HOST} — chat={CHAT_MODEL}, embed={EMBED_MODEL}")
    except httpx.ConnectError as e:
        raise RuntimeError(
            f"Cannot reach Ollama at {OLLAMA_HOST}. Start with: ollama serve\n{e}"
        ) from e


def parse_json_from_llm(text: str) -> dict[str, Any]:
    """Extract JSON object from LLM response (handles markdown fences)."""
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        text = fence.group(1).strip()
    return json.loads(text)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0


check_ollama()

## 2. WikiVault — full harness implementation

In [ ]:
@dataclass
class IndexEntry:
    title: str
    summary: str
    category: str


@dataclass
class LintIssue:
    kind: str
    message: str
    path: str | None = None


@dataclass
class WikiVault:
    root: Path

    @property
    def raw_dir(self) -> Path:
        return self.root / "raw"

    @property
    def wiki_dir(self) -> Path:
        return self.root / "wiki"

    @property
    def index_path(self) -> Path:
        return self.root / "index.md"

    @property
    def log_path(self) -> Path:
        return self.root / "log.md"

    @property
    def embeddings_path(self) -> Path:
        return self.root / ".embeddings.json"

    def ensure_structure(self) -> None:
        for sub in ("concepts", "entities", "sources", "synthesis", "derived"):
            (self.wiki_dir / sub).mkdir(parents=True, exist_ok=True)
        self.raw_dir.mkdir(parents=True, exist_ok=True)
        if not self.index_path.exists():
            self.index_path.write_text(
                "# Wiki Index\n\n## Concepts\n\n## Entities\n\n## Sources\n\n## Synthesis\n\n## Derived\n",
                encoding="utf-8",
            )
        if not self.log_path.exists():
            self.log_path.write_text(
                "# Wiki Log\n\nAppend-only timeline.\n", encoding="utf-8"
            )

    def read_index(self) -> list[IndexEntry]:
        entries: list[IndexEntry] = []
        category = "uncategorized"
        for line in self.index_path.read_text(encoding="utf-8").splitlines():
            if line.startswith("## "):
                category = line[3:].strip().lower()
                continue
            m = INDEX_ENTRY_RE.match(line.strip())
            if m:
                entries.append(IndexEntry(m.group(1), m.group(2), category))
        return entries

    def write_index(self, entries: list[IndexEntry]) -> None:
        by_cat: dict[str, list[IndexEntry]] = {}
        for e in entries:
            by_cat.setdefault(e.category, []).append(e)
        lines = ["# Wiki Index", ""]
        for cat in ("concepts", "entities", "sources", "synthesis", "derived"):
            lines += [f"## {cat.title()}", ""]
            for e in sorted(by_cat.get(cat, []), key=lambda x: x.title.lower()):
                lines.append(f"- [[{e.title}]] — {e.summary}")
            lines.append("")
        self.index_path.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")

    def append_log(self, action: str, detail: str) -> None:
        entry = f"\n## [{date.today().isoformat()}] {action} | {detail}\n"
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(entry)

    def list_wiki_pages(self) -> list[Path]:
        return sorted(self.wiki_dir.rglob("*.md"))

    def _slugify(self, text: str) -> str:
        s = re.sub(r"[^\w\s-]", "", text.lower().strip())
        return re.sub(r"[\s_-]+", "-", s).strip("-")

    def llm_extract_source(self, raw_path: Path) -> dict[str, Any]:
        """LLM call: read raw markdown, return structured extraction JSON."""
        raw_text = raw_path.read_text(encoding="utf-8")
        prompt = f"""You are a PKM wiki compiler. Read this source and extract structured knowledge.

Return ONLY valid JSON (no markdown) with this schema:
{{
  "title": "short source title",
  "summary": "1-2 sentence summary",
  "concepts": [{{"name": "...", "description": "1-2 sentences"}}],
  "entities": [{{"name": "...", "description": "1 sentence"}}]
}}

Extract 2-5 concepts and 0-3 entities. Be concise.

SOURCE:
{raw_text[:12000]}"""
        content = ollama_chat([{"role": "user", "content": prompt}])
        return parse_json_from_llm(content)

    def ingest_raw(self, raw_filename: str) -> list[Path]:
        """Ingest: LLM extraction → wiki pages → index → log."""
        raw_path = self.raw_dir / raw_filename
        if not raw_path.exists():
            raise FileNotFoundError(raw_path)

        print(f"LLM extracting from {raw_filename}...")
        data = self.llm_extract_source(raw_path)
        title = data["title"]
        summary = data["summary"]
        concepts = [(c["name"], c["description"]) for c in data.get("concepts", [])]
        entities = [(e["name"], e["description"]) for e in data.get("entities", [])]
        today = date.today().isoformat()
        created: list[Path] = []

        source_page = self.wiki_dir / "sources" / f"{self._slugify(title)}.md"
        body = f"---\nsource: raw/{raw_filename}\ningested: {today}\n---\n\n# {title}\n\n{summary}\n\n## Concepts\n\n"
        for name, desc in concepts:
            slug = self._slugify(name)
            cp = self.wiki_dir / "concepts" / f"{slug}.md"
            if cp.exists():
                text = cp.read_text(encoding="utf-8").rstrip()
                cp.write_text(text + f"\n\n## From [[{title}]]\n\n{desc}\n", encoding="utf-8")
            else:
                cp.write_text(f"# {name}\n\n{desc}\n\nSources: [[{title}]]\n", encoding="utf-8")
                created.append(cp)
            body += f"- [[{name}]] — {desc}\n"
        for name, desc in entities:
            ep = self.wiki_dir / "entities" / f"{self._slugify(name)}.md"
            if not ep.exists():
                ep.write_text(f"# {name}\n\n{desc}\n\nMentioned in: [[{title}]]\n", encoding="utf-8")
                created.append(ep)
        source_page.write_text(body, encoding="utf-8")
        created.append(source_page)

        entries = self.read_index()
        titles = {e.title.lower() for e in entries}
        new = [IndexEntry(title, summary[:80], "sources")]
        for name, desc in concepts:
            if name.lower() not in titles:
                new.append(IndexEntry(name, desc[:80], "concepts"))
        for name, desc in entities:
            if name.lower() not in titles:
                new.append(IndexEntry(name, desc[:80], "entities"))
        self.write_index(entries + new)
        self.append_log("ingest", title)
        print(f"Ingest complete: {title} ({len(concepts)} concepts, {len(entities)} entities)")
        return created

    def load_embeddings(self) -> dict[str, list[float]]:
        if self.embeddings_path.exists():
            return json.loads(self.embeddings_path.read_text(encoding="utf-8"))
        return {}

    def save_embeddings(self, store: dict[str, list[float]]) -> None:
        self.embeddings_path.write_text(json.dumps(store, indent=2), encoding="utf-8")

    def embed_all_pages(self) -> dict[str, list[float]]:
        """Embedding call: vectorize every wiki page (title + first 1500 chars)."""
        store = self.load_embeddings()
        pages = self.list_wiki_pages()
        to_embed: list[tuple[str, str]] = []
        for p in pages:
            key = str(p.relative_to(self.root))
            if key not in store:
                text = p.read_text(encoding="utf-8")[:1500]
                to_embed.append((key, f"{p.stem}\n{text}"))
        if to_embed:
            print(f"Embedding {len(to_embed)} page(s) via {EMBED_MODEL}...")
            vectors = ollama_embed([t for _, t in to_embed])
            for (key, _), vec in zip(to_embed, vectors):
                store[key] = vec
            self.save_embeddings(store)
        else:
            print("All pages already embedded.")
        return store

    def semantic_search(self, query: str, top_k: int = 3) -> list[tuple[str, float]]:
        """Embedding call: embed query, rank pages by cosine similarity."""
        store = self.load_embeddings()
        if not store:
            raise RuntimeError("No embeddings. Run embed_all_pages() first.")
        q_vec = np.array(ollama_embed([query])[0])
        scored = [
            (path, cosine_similarity(q_vec, np.array(vec)))
            for path, vec in store.items()
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def llm_answer(self, question: str, top_k: int = 3) -> str:
        """Query pipeline: semantic retrieval → LLM synthesis with citations."""
        hits = self.semantic_search(question, top_k=top_k)
        context_parts = []
        for rel_path, score in hits:
            p = self.root / rel_path
            context_parts.append(f"### [[{p.stem}]] (score={score:.3f})\n{p.read_text(encoding='utf-8')[:2000]}")
        context = "\n\n".join(context_parts)
        prompt = f"""Answer the question using ONLY the wiki pages below. Cite pages with [[Page Name]] wikilinks.

QUESTION: {question}

WIKI PAGES:
{context}"""
        print(f"LLM synthesizing answer from {len(hits)} retrieved page(s)...")
        return ollama_chat([{"role": "user", "content": prompt}])

    def lint(self) -> list[LintIssue]:
        issues: list[LintIssue] = []
        pages = self.list_wiki_pages()
        titles = {p.stem.replace("-", " ").title() for p in pages}
        index_titles = {e.title for e in self.read_index()}
        inbound: dict[str, int] = {t: 0 for t in titles}
        for p in pages:
            t = p.stem.replace("-", " ").title()
            for link in WIKILINK_RE.findall(p.read_text(encoding="utf-8")):
                if link not in titles and link not in index_titles:
                    issues.append(LintIssue("dead_link", f"Dead wikilink [[{link}]]", str(p)))
                elif link in inbound:
                    inbound[link] += 1
            if t not in index_titles:
                issues.append(LintIssue("missing_index", "Page not in index.md", str(p)))
        for t, count in inbound.items():
            if count == 0:
                rel = next((p for p in pages if p.stem.replace("-", " ").title() == t), None)
                if rel:
                    issues.append(LintIssue("orphan", f"No inbound links to [[{t}]]", str(rel)))
        return issues

## 3. Initialize vault & add raw source

In [ ]:
if VAULT_ROOT.exists():
    shutil.rmtree(VAULT_ROOT)

vault = WikiVault(VAULT_ROOT)
vault.ensure_structure()
print(f"Vault: {VAULT_ROOT}")

RAW_ARTICLE = """# Retrieval-Augmented Generation

RAG retrieves document chunks at query time and injects them into an LLM prompt.
The LLM rediscovers knowledge from scratch on every question — there is no accumulation.

Karpathy's compiled wiki pattern synthesizes at ingest time instead, building a
persistent markdown wiki with cross-references between raw sources and concept pages.

At scale, tools like qmd add hybrid BM25 + vector search when index.md outgrows
comfortable context limits. Local embeddings via Ollama (nomic-embed-text) keep
semantic search on-device for private vaults.
"""

raw_path = vault.raw_dir / "rag-vs-compiled-wiki.md"
raw_path.write_text(RAW_ARTICLE, encoding="utf-8")
print(f"Wrote raw source: {raw_path.name}")

## 4. Ingest — LLM extraction → wiki pages

In [ ]:
created = vault.ingest_raw("rag-vs-compiled-wiki.md")
print(f"\nCreated/updated {len(created)} page(s):")
for p in created:
    print(f"  {p.relative_to(VAULT_ROOT)}")

In [ ]:
print("=== index.md ===\n")
print(vault.index_path.read_text())
print("\n=== log.md ===\n")
print(vault.log_path.read_text())

## 5. Embed — Ollama embedding API

In [ ]:
embedding_store = vault.embed_all_pages()
print(f"\n{len(embedding_store)} page(s) in embedding index")
sample_key = next(iter(embedding_store))
print(f"Sample: {sample_key} → vector dim {len(embedding_store[sample_key])}")

## 6. Query — semantic search + LLM synthesis

In [ ]:
question = "How does a compiled wiki differ from RAG, and what role do local embeddings play?"

print("Semantic retrieval:")
for path, score in vault.semantic_search(question, top_k=3):
    print(f"  [{score:.3f}] {path}")

print("\n" + "=" * 60 + "\nLLM answer:\n" + "=" * 60)
answer = vault.llm_answer(question, top_k=3)
print(answer)

## 7. Lint — deterministic health check

In [ ]:
issues = vault.lint()
print(f"{len(issues)} issue(s):")
for issue in issues:
    loc = Path(issue.path).relative_to(VAULT_ROOT) if issue.path else "?"
    print(f"  [{issue.kind}] {issue.message} ({loc})")

vault.append_log("lint", f"{len(issues)} issues")

## 8. Final vault tree

In [ ]:
for path in sorted(VAULT_ROOT.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(VAULT_ROOT)} ({path.stat().st_size} bytes)")